# Week 1 - Foundations: Linear Systems & Eigen-Images for Recognition

Almost every method in this course reduces to two questions about a matrix:
*how do I solve* $A\mathbf{x} = \mathbf{b}$, and *what are the natural axes of my
data?* Week 1 builds both and fuses them into a classical recognition pipeline.
We solve a linear system two ways -- watching when the answer can be trusted --
then turn to the **eigen-image** idea: representing a whole library of images in
a compact basis of a few shared image-space variation patterns -- signed
directions of variation -- found by principal component analysis. The famous version is *eigenfaces*; here, on **real**
peripheral-blood-cell crops (BloodMNIST, via the course data layer), we build
**eigen-cells** and use them to compress, reconstruct, and classify.

The course's discipline is already here: never trust a number you haven't
checked. Every step is quantified -- the conditioning of the solve, the variance
a few modes capture, reconstruction error versus basis size, and classifier
accuracy on data it never saw during fitting.

**Reading.** Kutz, *Data-Driven Modeling & Scientific Computation*, 2nd ed. --
Chapter 2, sections 1--5 (direct and iterative solvers for $A\mathbf{x}=\mathbf{b}$,
steepest descent, eigenvalues and solvability, and eigen-decomposition for face
recognition) and Chapter 15, sections 1--5 (the singular value decomposition,
principal component analysis, and proper orthogonal modes -- the machinery behind
our eigen-images). Read them for the derivations; the treatment below is in our own
terms and runs against our own fixtures and the BloodMNIST library.

**Learning goals.**

- Solve a small linear system $A\mathbf{x}=\mathbf{b}$ directly and iteratively,
  and use the condition number to judge how much to trust the solution.
- Build an eigen-image basis by mean-centering an image library and taking its
  principal components with the singular value decomposition (`numpy.linalg.svd`).
- Read explained-variance ratios to decide how many modes reach 90/95/99% of
  the variance, and relate that to reconstruction error.
- Classify images in the compact eigen-basis with a nearest-neighbour rule,
  measured on the official held-out test partition, and state confidence honestly.


```{admonition} Which paradigm?
:class: note
**Data-driven.** You never write down a model of what a blood cell should look like. You mean-center the BloodMNIST microscopy crops, let the SVD hand you the eigen-cells -- the axes along which the images actually vary -- then reconstruct and classify in that learned basis with a nearest-neighbour rule. Those coordinates come inductively from the pixels, not from cell biology or the microscope's optics. That puts this week at the data-driven extreme of the course; next week makes the opposite, mechanistic move, committing to a dose-response model and fitting its parameters.
```


## Setup

We seed every random number generator and apply the course plotting style, so the
**numbers** below are reproducible from a cold kernel. (The eigen-image *displays*
carry one caveat, noted where they appear: each mode's sign is an arbitrary
convention, so a different linear-algebra backend may render an eigen-image with its
light and dark swapped without changing anything measurable.)

In [ ]:
# Colab setup: install the ddm4bio course library.
# No-op when ddm4bio is already importable (e.g. the course-site build), so
# this cell is safe everywhere. It is hidden from the rendered site via the
# "remove-cell" tag, but runs when this notebook is opened in Google Colab.
try:
    import ddm4bio  # noqa: F401
except ModuleNotFoundError:
    %pip install -q "ddm4bio @ git+https://github.com/symbiont-ai/ddm4bio.git"
    import ddm4bio  # noqa: F401

In [ ]:
import numpy as np

import ddm4bio
from ddm4bio import seed_everything
from ddm4bio.viz.style import set_style

seed_everything()
set_style()

print(f"ddm4bio version: {ddm4bio.__version__}")

## 1. Solving $A\mathbf{x}=\mathbf{b}$ two ways, and the role of conditioning

A linear system asks: which combination of the columns of $A$ reproduces the observation
$\mathbf{b}$? There are two broad strategies. A **direct** solver factors $A$ and then solves
by substitution; because our $A$ is symmetric positive-definite -- and we tell SciPy so with
`assume_a="pos"` -- it uses an SPD-specialized factorization (Cholesky) rather than general
Gaussian elimination / LU. An **iterative** solver instead starts from a guess and refines it --
here we use conjugate gradient, a standard choice for large sparse symmetric positive-definite
systems.

What decides whether either answer can be *trusted* is not the algorithm but the **conditioning**
of $A$. So we build two SPD systems of the same size but very different condition numbers -- one
well-conditioned, one badly so -- each with a *known* true solution, and measure what actually
matters: the **residual** $\|A\hat{\mathbf{x}} - \mathbf{b}\|$ (how well the answer satisfies
the equation), the **forward error** $\|\hat{\mathbf{x}} - \mathbf{x}_{\text{true}}\|$ (how
close it is to the truth), how a tiny nudge to $\mathbf{b}$ propagates, and how many
conjugate-gradient iterations each needs. Keeping both matrices SPD lets CG run on either.

In [ ]:
import inspect

from scipy.linalg import solve as direct_solve
from scipy.sparse.linalg import cg

# SciPy renamed the CG relative-tolerance keyword from `tol` to `rtol` in 1.12; use whichever
# this install exposes so the lesson runs on either version.
cg_tol_kw = "rtol" if "rtol" in inspect.signature(cg).parameters else "tol"


def spd_with_cond(n, cond, seed):
    """A symmetric positive-definite matrix with a prescribed condition number."""
    q, _ = np.linalg.qr(np.random.default_rng(seed).standard_normal((n, n)))
    eigenvalues = np.logspace(0.0, np.log10(cond), n)     # spread evenly from 1 to `cond`
    return (q * eigenvalues) @ q.T


def probe(A, x_true, noise_rel=1e-8):
    """Solve A x = b with a slightly noisy b, directly and by CG, and measure everything."""
    n = A.shape[0]
    b = A @ x_true
    noise = np.random.default_rng(1).standard_normal(n)
    noise *= noise_rel * np.linalg.norm(b) / np.linalg.norm(noise)   # small measurement-like noise
    x_hat = direct_solve(A, b + noise, assume_a="pos")              # solve the NOISY system
    x_clean = direct_solve(A, b, assume_a="pos")
    iters = [0]
    _, info = cg(A, b, maxiter=5 * n,
                 callback=lambda xk: iters.__setitem__(0, iters[0] + 1), **{cg_tol_kw: 1e-10})
    return {
        "cond": np.linalg.cond(A),
        "residual": np.linalg.norm(A @ x_hat - (b + noise)) / np.linalg.norm(b + noise),
        "forward": np.linalg.norm(x_hat - x_true) / np.linalg.norm(x_true),
        "rel_in": np.linalg.norm(noise) / np.linalg.norm(b),
        "rel_out": np.linalg.norm(x_hat - x_clean) / np.linalg.norm(x_clean),
        "cg_iters": iters[0],
        "cg_converged": info == 0,
    }


n = 100
x_true = np.random.default_rng(0).standard_normal(n)
A_well = spd_with_cond(n, cond=8.0, seed=42)      # well-conditioned SPD
A_ill = spd_with_cond(n, cond=1e8, seed=43)       # ill-conditioned SPD (still SPD, so CG applies)
print(f"Built two {n}x{n} SPD systems: cond {np.linalg.cond(A_well):.0f} vs {np.linalg.cond(A_ill):.0e}, each with a known true solution.")

We now solve each system with a *slightly noisy* right-hand side -- the kind of small
measurement or round-off error real data always carries -- and read off six numbers. Two of
them are easy to confuse: the **residual** measures how well our answer satisfies the equation
we actually solved, while the **forward error** measures how far that answer is from the true
solution. Watch the gap between them open up as the conditioning worsens.

In [ ]:
for label, A in [("Well-conditioned", A_well), ("Ill-conditioned", A_ill)]:
    r = probe(A, x_true)
    cg_note = (f"{r['cg_iters']} (converged)" if r["cg_converged"]
               else f"{r['cg_iters']} (did NOT converge)")
    print(f"\n{label}  (cond = {r['cond']:.1e}):")
    print(f"  relative residual  ||A x_hat - b|| / ||b||        : {r['residual']:.1e}")
    print(f"  forward error      ||x_hat - x_true|| / ||x_true|| : {r['forward']:.1e}")
    print(f"  input perturbation {r['rel_in']:.1e} -> output perturbation {r['rel_out']:.1e}"
          f"  (amplification {r['rel_out'] / r['rel_in']:.1e}x, bounded by cond)")
    print(f"  conjugate-gradient iterations to 1e-10             : {cg_note}")

**QC note.** Look at the two systems side by side. Both were "solved" to a relative residual
at machine precision -- the computed answer satisfies its (noisy) equation about as well as
floating point allows. Yet on the well-conditioned system the forward error is tiny too, while
on the ill-conditioned one it is enormous: a solution that nails the equation to ~15 digits is
still *wrong in the second decimal place*. **A small residual does not imply an accurate
solution** -- it only says the answer is consistent with the data you fed in, and an
ill-conditioned matrix turns the small, unavoidable error in that data into a large error in
the answer (the amplification is bounded by the condition number, though a single perturbation
rarely saturates that bound). Conjugate gradient tells the same story from the other side: a
few dozen iterations on the well-conditioned system, but no convergence on the ill-conditioned
one. The lesson for the rest of the course: before trusting any solve, check the condition
number -- the residual alone will not warn you.

## 2. From pixels to an image library

We now switch from a single linear system to a *library* of images. Here the
library is **real**: we pull BloodMNIST -- peripheral-blood-cell microscopy
crops from MedMNIST v2 -- through the course data layer,
`get_dataset("bloodmnist")`. Each crop is a small RGB image carrying an integer
cell-type label. We convert every crop to grayscale (averaging the colour
channels), flatten it to a vector, and take a *seeded* few-hundred-image
subsample so the whole lesson runs briskly. (The **eigen-cells** themselves come next, once we apply PCA to this library.)
The loader returns the real data whenever it can reach its source; only with no network
at all does it fall back to a smaller, clearly labelled stand-in from a *different* public
dataset (`sklearn`'s 8x8 handwritten digits). The stand-in shares only the payload
**schema** -- the same dict keys and an `(N, H, W, C)` uint8 layout with train/val/test
splits -- so the same adaptive code runs end-to-end either way. It is *not* the same shape
or content: the stand-in is 8x8x1 with ten classes, versus the real 28x28x3 crops with
eight cell types. We print which one we got; the deployed lesson and the numbers quoted
below use the real data.

In [ ]:
from ddm4bio.datasets import get_dataset

ds = get_dataset("bloodmnist", seed=0)
print(f"Data source : {ds.source}")
print(f"Provenance  : {ds.provenance}")

# BloodMNIST ships an OFFICIAL train / validation / test split (a 7:1:2 partition), so we
# use it rather than re-splitting: the training partition drives the eigen-basis and the
# classifier, and the official test partition is used only to score, never to fit. We take a
# seeded subsample of each so the lesson runs briskly, and average the colour axis to grayscale.
rng_sub = np.random.default_rng(0)


def take(images, labels, n):
    idx = rng_sub.choice(images.shape[0], size=min(n, images.shape[0]), replace=False)
    gray = images[idx].mean(axis=-1)                       # (n, H, W) grayscale
    return gray, gray.reshape(gray.shape[0], -1).astype(float), labels[idx].ravel()


images, X, y = take(ds.payload["train_images"], ds.payload["train_labels"], 400)
# TEST: the FULL official test partition. 1-NN scoring never refits, so evaluating on all
# of it (rather than a 400-image subsample) costs almost nothing and tightens the accuracy
# interval; only the TRAIN library that builds the basis needs the seeded few-hundred cap.
test_gray = ds.payload["test_images"].mean(axis=-1)
X_test = test_gray.reshape(test_gray.shape[0], -1).astype(float)
y_test = ds.payload["test_labels"].ravel()
img_h, img_w = images.shape[1], images.shape[2]
classes = np.unique(np.concatenate([y, y_test]))
n_classes = classes.size

print(f"Training library : {X.shape[0]} images of {img_h}x{img_w} pixels ({n_classes} cell classes)")
print(f"Official test set: {X_test.shape[0]} held-out images (the full official test partition, used only to score)")
print(f"Flattened feature matrix X: {X.shape} (samples x pixels)")

# BloodMNIST publishes named cell types (MedMNIST v2; Acevedo et al., 2020). We label the
# real crops with these; the offline fallback is synthetic, so it keeps generic class ids.
BLOODMNIST_CLASSES = {
    0: "basophil", 1: "eosinophil", 2: "erythroblast", 3: "immature granulocyte",
    4: "lymphocyte", 5: "monocyte", 6: "neutrophil", 7: "platelet",
}


def class_label(i):
    return BLOODMNIST_CLASSES[int(i)] if ds.source == "real" else f"class {int(i)}"

A quick look at a handful of raw "cells" from the library. This is what the
recognizer has to work with -- low-resolution, variable, and noisy.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 6, figsize=(11, 3.4))
for ax, img, label in zip(axes.flat, images, y):
    ax.imshow(img, cmap="gray_r")
    ax.set_title(class_label(label), fontsize=7)
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle(f"Twelve crops from the {img_h}x{img_w} blood-cell library")
fig;

Before modeling, one look at the **class balance** of the training library. The eight cell
types are not equally represented -- worth keeping in mind when we read the 1-NN confusion
matrix later, since a nearest-neighbour rule leans toward the crowded classes and struggles
on the rare ones.

In [ ]:
train_classes, train_counts = np.unique(y, return_counts=True)
order = np.argsort(train_counts)[::-1]                     # most common first
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(range(len(order)), train_counts[order], color='0.6', edgecolor='white')
ax.bar_label(bars, padding=2, fontsize=8)
ax.set_xticks(range(len(order)))
ax.set_xticklabels([class_label(train_classes[i]) for i in order],
                   rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Training images')
ax.set_title(f'Class balance of the {y.size}-image training library')
ax.margins(y=0.12)
fig;

## 3. Building the eigen-image basis (the ground-truth-adjacent check)

The eigen-image recipe is exactly PCA on the image library:

1. **Mean-center** -- subtract the average image so the basis describes
   *deviations* from the mean, not the mean itself.
2. **Take principal components** -- the right singular vectors of the centered
   data matrix are the eigen-images: orthogonal full-resolution pixel patterns (one weight per
   image pixel), ordered by how much library variance each explains.
3. **Project** -- every image becomes a short vector of coordinates in this
   basis (its PCA scores).

Because PCA is a *deterministic* linear algebra operation (an SVD), its "ground
truth" is self-checking: the explained-variance ratios are guaranteed to be
non-negative and to sum to one, and the top modes must reconstruct the data
better than any other orthogonal basis of the same size. We verify those
invariants explicitly rather than take them on faith.

In [ ]:
# Mean image and centered library.
mean_image = X.mean(axis=0)
X_centered = X - mean_image

# The eigen-images ARE the principal components: the right singular vectors of the centered
# library. NumPy's SVD returns them directly, ordered by how much variance each explains.
_, singular_values, vt = np.linalg.svd(X_centered, full_matrices=False)
evr = singular_values**2 / np.sum(singular_values**2)   # explained-variance ratio per mode

# Mean-centering removes one degree of freedom (the centered columns sum to zero over the
# samples), so the library has at most n-1 non-trivial modes: the final singular value is
# numerically zero and its direction is arbitrary. We count the modes that actually carry
# signal with the same tolerance rule np.linalg.matrix_rank uses -- but from the singular
# values already in hand, rather than paying for a second SVD.
rank_tol = singular_values[0] * max(X_centered.shape) * np.finfo(singular_values.dtype).eps
effective_rank = int(np.sum(singular_values > rank_tol))

print(f"Explained-variance ratios sum to 1: {np.isclose(evr.sum(), 1.0)}")
print(f"All ratios non-negative and non-increasing: "
      f"{np.all(evr >= 0) and np.all(np.diff(evr) <= 1e-12)}")
print(f"Variance captured by mode 1 alone : {evr[0]:.1%}")
print(f"Variance captured by top 10 modes : {evr[:10].sum():.1%}")
print(f"Effective rank (modes above tol)  : {effective_rank} of {vt.shape[0]} "
      f"(centering annihilates one direction)")

The scree curve shows how fast the explained variance decays. Unlike the sharp
rank-2 elbow of a purely synthetic fixture, real image data has a *gentle*
shoulder: a handful of modes dominate, but a long tail of small modes carries
finer, lower-variance variation. What that tail *contains* is not settled by the
spectrum alone -- some may be class-relevant morphology, some may be nuisance
variation (illumination, staining, segmentation) or noise; the plot tells us only
that each of these modes carries little variance.

In [ ]:
from ddm4bio.viz.plots import scree_plot

ax = scree_plot(evr[:20])       # first 20 modes; the tail is a slow decay to zero
ax.set_title("Scree plot: explained variance of the top 20 eigen-images")
ax.figure;

Every reconstruction in this section starts from one picture: the **mean cell**, the
pixel-wise average of the training library. PCA writes each image as this mean *plus* a
weighted sum of the eigen-images that follow, so it is worth seeing on its own first.

In [ ]:
fig, ax = plt.subplots(figsize=(3.3, 3.3))
im = ax.imshow(mean_image.reshape(img_h, img_w), cmap='gray_r')
ax.set_xticks([]); ax.set_yticks([])
ax.set_title('Mean cell (library average)')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='mean intensity')
fig;

Now the eigen-images themselves -- the *signed directions of variation* in image
space, not template or prototype cells. Each is a full-resolution pixel pattern
reshaped back to the image grid. The first few look like smooth blobs that capture
gross cell shape and brightness; later ones encode progressively finer,
higher-frequency contrasts. Any library image is a weighted sum of the mean image
plus these patterns. One caveat on reading them: each mode's overall sign is an
arbitrary SVD convention -- flip a pattern's light and dark together and flip its
coordinate for every image, and nothing observable changes -- and it can differ
across linear-algebra backends. Treat the *structure* of each eigen-image as
meaningful, not its polarity.

In [ ]:
n_show = 8
eigen_images = vt[:n_show]                     # top-n signed variation patterns (rows of Vt)
vmax = float(np.abs(eigen_images).max())       # ONE scale, symmetric about zero, for all panels
ncols = 4
nrows = int(np.ceil(n_show / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(2.1 * ncols, 2.35 * nrows),
                         constrained_layout=True)
for i, ax in enumerate(axes.ravel()):
    if i < n_show:
        im = ax.imshow(eigen_images[i].reshape(img_h, img_w),
                       cmap='RdBu_r', vmin=-vmax, vmax=vmax)   # symmetric -> 0 maps to white
        ax.set_title(f'PC {i + 1}   ({evr[i]:.1%})', fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Top 8 eigen-images ('eigen-cells'), each labelled with its explained variance")
fig.colorbar(im, ax=axes, shrink=0.85, fraction=0.05, pad=0.02,
             label='signed loading  (blue < 0 < red)')
fig;

## 4. Reconstruction error vs. number of modes

How many eigen-images do we actually need? Project each image onto the top $k$
modes, reconstruct it, and measure the error. As $k$ grows the reconstruction
tightens; the useful question is where the curve flattens -- the point past which
extra modes buy little fidelity. We report a single **global (Frobenius) relative
error** over the whole library, $\lVert X_c - \hat{X}_c\rVert_F / \lVert X_c\rVert_F$ --
one number for the entire sample-by-pixel matrix, *not* an average of per-image
errors -- and overlay the variance-captured milestones (90/95/99%). Because it is the
global Frobenius error, it obeys an *exact* identity with the variance spectrum, which
we check below; we then look at how the error is distributed across individual images.

In [ ]:
def reconstruct_with_k(X_centered, vt, k):
    """Project onto the top-k eigen-images and map back to pixel space."""
    basis = vt[:k]                      # (k, n_pixels)
    scores = X_centered @ basis.T       # (n_samples, k)
    return scores @ basis               # (n_samples, n_pixels), centered reconstruction


def global_frobenius_error(reference, approx):
    """One relative error for the WHOLE sample-by-pixel matrix: ||ref - approx||_F / ||ref||_F.

    This is a global (energy-weighted) Frobenius error, not the average of per-image relative
    errors -- high-norm images count for more. The next cell looks at the per-image spread.
    """
    return float(np.linalg.norm(reference - approx) / np.linalg.norm(reference))


# Variance milestones first, so the mode sweep and its plot span the real basis.
cum_evr = np.cumsum(evr)


def modes_for(threshold):
    return int(np.searchsorted(cum_evr, threshold) + 1)


k90, k95, k99 = modes_for(0.90), modes_for(0.95), modes_for(0.99)
n_modes = effective_rank            # the modes that carry signal (Section 3), not vt.shape[0]

# A sweep that spans the whole basis -- adapting to the real 28x28 data (hundreds of modes)
# and the 8x8 offline fallback alike -- and always includes the variance milestones.
base = [1, 2, 4, 8, 16, 32, 64, 128, 256]
k_values = sorted({k for k in base if k < n_modes} | {k90, k95, k99, n_modes})
errors = [global_frobenius_error(X_centered, reconstruct_with_k(X_centered, vt, k))
          for k in k_values]

# Self-check: for the GLOBAL Frobenius error, reconstruction error is not merely correlated
# with explained variance -- it equals the square root of the leftover variance exactly,
#     ||X_c - X_c^(k)||_F / ||X_c||_F  ==  ||s[k:]|| / ||s||.
# We form the right-hand side from the TAIL of the singular values. (The tempting form
# sqrt(1 - cum_evr[k-1]) is algebraically identical but loses every digit to catastrophic
# cancellation near full rank -- 1 minus a sum that already equals ~1 -- the very trap
# Section 1 warned about, so we avoid it here.)
s_norm = np.linalg.norm(singular_values)
tail_error = [float(np.linalg.norm(singular_values[k:]) / s_norm) for k in k_values]
assert np.allclose(errors, tail_error, rtol=0, atol=1e-12), \
    "reconstruction error must equal sqrt(leftover variance)"

print(f"Modes to reach 90% variance: {k90}")
print(f"Modes to reach 95% variance: {k95}")
print(f"Modes to reach 99% variance: {k99}  (of {n_modes} effective modes)")
print("Self-check passed: global Frobenius error == sqrt(leftover variance) for every k.")
for k, e in zip(k_values, errors):
    print(f"  k={k:3d}:  global Frobenius reconstruction error = {e:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(k_values, errors, marker="o", linewidth=1.5)
ax.set_xscale("log")
for (k, name), y_lab in zip([(k90, "90%"), (k95, "95%"), (k99, "99%")], [0.62, 0.40, 0.62]):
    ax.axvline(k, color="0.6", linestyle="--", linewidth=1)
    ax.text(k, y_lab, f"{name}\n(k={k})", fontsize=8, color="0.35", ha="center",
            backgroundcolor="white")
ax.set_xlabel("Number of eigen-images (k, log scale)")
ax.set_ylabel("Global (Frobenius) reconstruction error")
ax.set_title("Reconstruction error falls as the eigen-basis grows")
fig;

That curve is one number per $k$ for the entire library -- but the global Frobenius
error hides real spread across images. At the 95% milestone some crops reconstruct far
better than others. The histogram below is the *per-image* relative error at $k=k_{95}$:
the distribution the single global number summarizes.

In [ ]:
k_dist = k95
recon_k = reconstruct_with_k(X_centered, vt, k_dist)
per_image_err = (np.linalg.norm(X_centered - recon_k, axis=1)
                 / np.linalg.norm(X_centered, axis=1))
global_err = global_frobenius_error(X_centered, recon_k)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(per_image_err, bins=30, color="0.6", edgecolor="white")
ax.axvline(global_err, color="C3", linestyle="--", linewidth=1.5,
           label=f"global Frobenius = {global_err:.3f}")
ax.axvline(per_image_err.mean(), color="C0", linestyle=":", linewidth=1.5,
           label=f"mean per-image = {per_image_err.mean():.3f}")
ax.set_xlabel(f"Per-image relative L2 reconstruction error at k={k_dist}")
ax.set_ylabel("Number of images")
ax.set_title(f"Reconstruction quality varies across images (k={k_dist}, 95% variance)")
ax.legend(fontsize=8)
fig;
print(f"Per-image relative error at k={k_dist}: "
      f"median {np.median(per_image_err):.3f}, "
      f"90th pct {np.quantile(per_image_err, 0.9):.3f}, "
      f"worst {per_image_err.max():.3f}  "
      f"(global Frobenius {global_err:.3f}, mean per-image {per_image_err.mean():.3f})")

A visual confirmation: the same cell crop reconstructed from an increasing
number of modes. With only a few eigen-images it is a smudge; it takes on the
order of a hundred modes to sharpen on this real library, and past the 99%
variance milestone the extra modes change little.

In [ ]:
sample_idx = 0
ks_to_show = sorted({1, 8, k95, k99, n_modes})
fig, axes = plt.subplots(1, len(ks_to_show) + 1, figsize=(2.0 * (len(ks_to_show) + 1), 2.2))
axes[0].imshow(images[sample_idx], cmap="gray_r")
axes[0].set_title("original", fontsize=9)
axes[0].set_xticks([]); axes[0].set_yticks([])
for ax, k in zip(axes[1:], ks_to_show):
    recon = reconstruct_with_k(X_centered, vt, k)[sample_idx] + mean_image
    ax.imshow(recon.reshape(img_h, img_w), cmap="gray_r")
    ax.set_title(f"k={k}", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle(f"Reconstructing one blood-cell crop ({class_label(y[sample_idx])}) from k eigen-images")
fig;

### Dimensionality reduction is not file compression

It is tempting to say the eigen-basis "compresses" a 784-pixel image to $k$ numbers. That
is true as *coordinate dimensionality per image* -- each image becomes a length-$k$ vector --
but it is **not** the same as shrinking the bytes on disk. To rebuild any image you must also
keep the shared mean image and the $k$ basis vectors, so the honest accounting for a library
of $N$ images of $p$ pixels, each number stored in $b$ bytes, is

$$\text{encoded bytes} \;=\; \underbrace{p\,b}_{\text{mean}} \;+\; \underbrace{k\,p\,b}_{\text{basis}} \;+\; \underbrace{N\,k\,b}_{\text{coefficients}},$$

against a raw raster of $N\,p$ bytes at one byte per 8-bit pixel. The mean and basis are a
*fixed overhead* paid once, so a saving appears only when $N$ is large enough to amortize
them -- and because the coefficients here are floats while the raw pixels are single bytes,
the per-image cost can even run the wrong way. We compute it for our own library.

In [ ]:
n_pixels = X.shape[1]          # p, pixels per image (784)
k_enc = k95                    # the operating point the classifier uses
n_images = X.shape[0]          # N, images encoded in the basis
b_float = 4                    # float32 bytes for mean, basis, coefficients
b_raw = 1                      # uint8 bytes per pixel for a natural 8-bit raster

mean_bytes = n_pixels * b_float
basis_bytes = k_enc * n_pixels * b_float
coeff_bytes = n_images * k_enc * b_float
encoded_bytes = mean_bytes + basis_bytes + coeff_bytes
raw_uint8 = n_images * n_pixels * b_raw
raw_float32 = n_images * n_pixels * b_float

per_image_saving = n_pixels * b_raw - k_enc * b_float      # uint8 pixel vs float32 coefficient
n_star = (mean_bytes + basis_bytes) / per_image_saving if per_image_saving > 0 else np.inf
breakeven_images = int(np.ceil(n_star)) if np.isfinite(n_star) else None


def kib(n):
    return n / 1024


print(f"Storing {n_images} images of {n_pixels} pixels; eigen-basis at k={k_enc} (float32):")
print(f"  mean image        : {mean_bytes:>10,d} B")
print(f"  basis, {k_enc} modes  : {basis_bytes:>10,d} B   <- fixed overhead, paid once")
print(f"  {n_images} x {k_enc} coeffs : {coeff_bytes:>10,d} B")
print(f"  encoded TOTAL     : {encoded_bytes:>10,d} B   ({kib(encoded_bytes):.0f} KiB)")
print(f"  raw uint8 raster  : {raw_uint8:>10,d} B   ({kib(raw_uint8):.0f} KiB)")
verdict = "smaller" if encoded_bytes < raw_uint8 else "LARGER -- no compression here"
print(f"  encoded / raw uint8   = {encoded_bytes / raw_uint8:.0%}   ({verdict})")
print(f"  encoded / raw float32 = {encoded_bytes / raw_float32:.0%}   (same-precision comparison)")
if breakeven_images is not None:
    print(f"  break-even vs the uint8 raster: N = {breakeven_images} images "
          f"(this library holds {n_images})")
else:
    print("  float32 coefficients never beat an 8-bit raster per image, at any N")

# Total storage vs library size, the two ways -- the crossover is the amortization point.
n_grid = np.arange(1, 2001)
enc_kib = kib(mean_bytes + basis_bytes + n_grid * k_enc * b_float)
raw_kib = kib(n_grid * n_pixels * b_raw)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(n_grid, enc_kib, label=f"eigen-encoded (k={k_enc}, float32)")
ax.plot(n_grid, raw_kib, label="raw uint8 raster")
ax.axvline(n_images, color="0.5", linestyle=":", linewidth=1.2, label=f"our library (N={n_images})")
if breakeven_images is not None:
    ax.axvline(breakeven_images, color="C2", linestyle="--", linewidth=1.2,
               label=f"break-even (N={breakeven_images})")
ax.set_xlabel("Number of images stored, N")
ax.set_ylabel("Total storage (KiB)")
ax.set_title("Basis overhead must be amortized before the eigen-code saves space")
ax.legend(fontsize=8)
fig;

## 5. Recognition on the official held-out test set

The payoff: classification in the compact basis, scored honestly. BloodMNIST ships an
official train / test split, so we use it rather than re-splitting the training images: we
learn the eigen-basis, the mean image, and the mode count from the **training** partition
alone, then classify each image of the **official test** partition by its nearest training
neighbour. No preprocessing, basis, mode count, or classifier ever sees the test set.

A caveat worth stating plainly: this bounds *memorization*, not every optimistic bias. The
check below rules out pixel-identical crops crossing the split, but without donor or
acquisition identifiers we cannot rule out the same donor, near-duplicate crops, or shared
staining and imaging conditions spanning the two partitions -- so we call the estimate an
honest held-out one, not "leakage-free."

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Sanity check across the official boundary: no test crop is byte-identical to a training
# crop. This rules out pixel-identical duplicates only -- the interpretation block below
# states what it cannot rule out.
train_rows = {row.tobytes() for row in X}
n_dup = sum(row.tobytes() in train_rows for row in X_test)
print(f"Training library: {X.shape[0]} images   Official test set: {X_test.shape[0]} images")
print(f"Test crops byte-identical to a training crop: {n_dup}")

# The eigen-basis, mean, and mode count were all fit on the TRAINING partition (Sections
# 3-4); k is 95% of the training-library variance, chosen without ever touching the test set.
k_class = k95
Z_train = (X - mean_image) @ vt[:k_class].T
Z_test = (X_test - mean_image) @ vt[:k_class].T   # TRAIN mean & basis applied to the test set
print(f"Classifying in a {k_class}-dimensional eigen-basis (down from {X.shape[1]} raw pixels).")

In [ ]:
knn = KNeighborsClassifier(n_neighbors=1)
knn.fit(Z_train, y)
acc_eigen = knn.score(Z_test, y_test)

# Baseline: the same 1-NN rule on raw pixels, for an honest comparison.
knn_raw = KNeighborsClassifier(n_neighbors=1)
knn_raw.fit(X, y)
acc_raw = knn_raw.score(X_test, y_test)

print(f"1-NN accuracy in the {k_class}-mode eigen-basis : {acc_eigen:.3f}")
print(f"1-NN accuracy on raw {X.shape[1]} pixels (baseline)   : {acc_raw:.3f}")
print(f"Dimensionality reduction: {X.shape[1]} -> {k_class} "
      f"({100 * k_class / X.shape[1]:.0f}% of the features), "
      f"accuracy change {acc_eigen - acc_raw:+.3f}")

The eigen-basis accuracy essentially matches the raw-pixel baseline -- and that is the
*point*, not a disappointment. PCA chooses its axes to capture **variance**, which is not
the same as **class separation**: the leading modes describe how the images vary, not how
the cell types differ, so a compact PCA basis *preserves* the classification signal rather
than sharpening it. A big accuracy jump here would actually be suspicious. When the goal is
classification *and* labels are available, the right move is to project onto
class-discriminating axes instead -- linear discriminant analysis -- which the course takes
up later. Here PCA earns its keep by **compression**, not by beating raw pixels.

The confusion matrix shows *where* the recognizer struggles. Rather than assume a pattern,
we print the largest off-diagonal cells below and read them directly: any clustering among
cell types with similar *grayscale* morphology is a hypothesis to check against the matrix,
not a given -- and recall we discarded colour and staining cues when we converted to
grayscale, so the recognizer cannot be confusing cells on those. This class-resolved view is
exactly what a single accuracy number hides.

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, knn.predict(Z_test), labels=classes)
fig, ax = plt.subplots(figsize=(6.8, 5.8))
im = ax.imshow(cm, cmap="Blues")
ax.set_xlabel("Predicted cell type")
ax.set_ylabel("True cell type")
ax.set_title(f"1-NN confusion matrix in the eigen-basis (k={k_class})")
ax.set_xticks(range(n_classes)); ax.set_yticks(range(n_classes))
ax.set_xticklabels([class_label(c) for c in classes], rotation=45, ha="right", fontsize=7)
ax.set_yticklabels([class_label(c) for c in classes], fontsize=7)
for i in range(n_classes):
    for j in range(n_classes):
        if cm[i, j]:
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    fontsize=7, color="0.2" if cm[i, j] < cm.max() / 2 else "white")
fig.colorbar(im, ax=ax, fraction=0.046, label="count")
fig;

# Show the largest off-diagonal confusions so the "where it struggles" reading is
# demonstrated, not asserted.
off = cm.copy()
np.fill_diagonal(off, 0)
flat = np.argsort(off.ravel())[::-1]
print("Largest confusions (true -> predicted : count):")
for f in flat[:5]:
    i, j = np.unravel_index(f, off.shape)
    if off[i, j] == 0:
        break
    print(f"  {class_label(classes[i])} -> {class_label(classes[j])} : {off[i, j]}")

**Why the held-out test set matters.** We fit the eigen-basis, the mean image, the number of
modes, and the classifier using only the training partition, then evaluated on the official
test partition, which none of that fitting had touched. The reported accuracy is therefore an
estimate of performance on *new* images, not a memorization score. Fitting the representation on the
full (pooled train-and-test) data can bias the test estimate, often optimistically, because
information from the test distribution enters the fitted pipeline -- though on any single
finite split a contaminated analysis can move either way.

In [ ]:
from ddm4bio.interpret import interpretation_block, show_interpretation

acc_se = float(np.sqrt(acc_eigen * (1.0 - acc_eigen) / y_test.size))
acc_lo, acc_hi = acc_eigen - 1.96 * acc_se, acc_eigen + 1.96 * acc_se

# Scoped to THIS section's claim (the classifier): one plain sentence, the confidence and the
# single reason it is not higher, and the two caveats that bear on it. The scree-tail, storage,
# and compression points live in Sections 3-4 where they were made -- not re-dumped here.
block = interpretation_block(
    claim=(
        f"In a {k_class}-mode eigen-basis ({100 * k_class / X.shape[1]:.0f}% of the pixels), a "
        f"1-nearest-neighbour classifier matches raw-pixel accuracy on held-out cells "
        f"({acc_eigen:.2f} vs {acc_raw:.2f}): the compact representation keeps the recognition "
        "signal rather than sharpening or losing it."
    ),
    confidence="moderate",
    limitations_list=[
        "Grayscale only -- we discarded the colour and staining cues that real blood-cell "
        "typing relies on.",
        "1-NN is a deliberately simple recognizer chosen for transparency, not a strong one.",
    ],
    evidence=(
        f"The estimate is precise (95% CI [{acc_lo:.2f}, {acc_hi:.2f}] on {y_test.size} held-out "
        "images, all fitting done on the training split), but BloodMNIST ships no donor "
        "identifiers, so we cannot rule out the same patient's cells in both splits -- which "
        "would flatter the score"
    ),
)
show_interpretation(block)

## Exercises

Your graded work for this week is **Problem Set 1 (PS1) -- "The Eigen-Subspace as a
Model of Normal Cells"**, distributed and auto-graded through GitHub Classroom. It keeps
this lesson's eigen-image basis but turns it to a new question: if the top principal axes
capture what a *normal* cell looks like, what does the **residual** -- the part that does not
fit -- tell you? The basis primitives (`eigen_basis`, `project`, `reconstruct`) and the data
loader are provided; you fill in the analysis on real BloodMNIST.

- **Part A -- denoise by low-rank projection.** A clean image lives in a few principal axes
  while additive noise spreads across all of them, so projecting a noisy crop onto the top-*k*
  subspace and reconstructing keeps the signal and discards most of the noise -- but only at
  the right rank. Implement `snr_db`, `denoise`, and `best_rank_for_denoising`, and explain
  why the SNR-versus-rank curve rises, peaks, and falls.
- **Part B -- flag out-of-QC images by reconstruction error.** An image that does not belong
  to the normal subspace reconstructs badly, so its reconstruction error is a novelty score
  for catching corrupted acquisitions that add out-of-subspace structure (sensor noise,
  saturation/clipping, debris). Implement
  `reconstruction_anomaly_score`, `detection_auc` (the detector's ROC-AUC), and
  `flag_threshold` (a false-alarm-bounded cutoff), and close with an interpretation block at a
  defensible confidence level.

Refer to the PS1 repository README for the submission and auto-grading details.